## 1. Business Core: Configuration & Pricing
At the heart of the system lies a flexible pricing structure. By centralizing the product catalog and pricing logic, we ensure the system is both maintainable and scalable.

### Data Infrastructure
- **`PRICE_LIST`**: A centralized dictionary defining the baseline rates for all services. This allows for rapid market adjustments without modifying core algorithms.
- **`calculate_order_total(order_dict)`**: A robust function that aggregates custom service baskets. It transforms complex customer requests (e.g., "2 sofas and a carpet") into a single, actionable `base_price` metric, serving as the input for our pricing engine.

*Key Benefit: Decoupling business rules from predictive modeling, allowing for agile business operations.*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

PRICE_LIST = {
    'sofa_2': 350,
    'sofa_3': 450,
    'sofa_l': 700,
    'mattress_single': 200,
    'mattress_double': 350,
    'carpet': 200
}

def calculate_order_total(order_dict):
    total = 0
    for item, quantity in order_dict.items():
        if item in PRICE_LIST:
            total += PRICE_LIST[item] * quantity
        else:
            print(f"שים לב: המוצר {item} לא קיים במחירון!")
    return total

## 2. Data Simulation & Feature Engineering
To train a predictive model effectively, we built a synthetic dataset that mimics the daily operations of a cleaning service business. This dataset incorporates real-world temporal and behavioral variables:

### Key Features
- **Temporal Dynamics**: We engineered features such as `month` and `is_weekend` to capture seasonality and cyclical demand patterns.
- **Market Demand Modeling**: The `demand` variable is intelligently constructed using a baseline distribution, boosted by summer peaks (months 6–8) and weekend spikes.
- **Probabilistic Modeling**: The `was_booked` target variable is not random; it is derived from a logistic-style probability function where purchase likelihood is negatively correlated with `base_price` and positively correlated with `is_holiday` and `is_weekend` effects.

*Key Benefit: By simulating realistic market shocks, we ensure the model learns the underlying relationship between price, demand, and conversion, rather than just overfitting to noise.*

In [2]:
n_days = 365
dates = [datetime(2025, 1, 1) + timedelta(days=i) for i in range(n_days)]
df = pd.DataFrame({'date': dates})

df['month'] = df['date'].dt.month
df['is_weekend'] = df['date'].dt.weekday >= 5
df['area'] = np.random.choice(['Center', 'North', 'South'], n_days)
df['service_type'] = np.random.choice(list(PRICE_LIST.keys()), n_days)
df['demand'] = np.random.randint(5, 15, size=n_days) + (df['month'].apply(lambda x: 10 if 6 <= x <= 8 else 0)) + (df['is_weekend'] * 3)
df['base_price'] = np.random.choice([200, 300, 400], n_days)

df['is_holiday'] = np.random.choice([0, 1], n_days, p=[0.9, 0.1])
prob = 0.8 - (df['base_price'] / 1000) + (df['is_holiday'] * 0.3) + (df['is_weekend'] * 0.2)
df['was_booked'] = (np.random.rand(n_days) < prob).astype(int)

## 3. Predictive Modeling Strategy
We employed a Gradient Boosting framework to predict customer conversion probability, prioritizing both predictive accuracy and interpretability.

### Technical Implementation
- **Feature Encoding**: We utilized One-Hot Encoding (`pd.get_dummies`) to transform categorical features (such as `area` and `service_type`) into a high-dimensional space suitable for machine learning.
- **Handling Class Imbalance**: In real-world service businesses, 'no-shows' or non-conversions often outweigh actual bookings. We applied `compute_sample_weight(class_weight='balanced')` to adjust the model's sensitivity, ensuring it treats potential conversion opportunities with equal importance to non-conversions.
- **Model Architecture**: We chose the `XGBClassifier` for its superior performance on tabular data, utilizing 100 boosting rounds to learn the non-linear relationship between our input features and the target variable.



*Key Benefit: The model acts as a reliable decision-support tool, providing granular probability scores that allow us to optimize pricing strategies dynamically.*

In [3]:
df_dummies = pd.get_dummies(df.drop('date', axis=1))
X = df_dummies.drop('was_booked', axis=1)
y = df_dummies.was_booked

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
weights = compute_sample_weight(class_weight='balanced', y=y_train)
model_xgbc = XGBClassifier(n_estimators=100, learning_rate=0.1)
model_xgbc.fit(X_train, y_train, sample_weight=weights)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

## 4. Dynamic Pricing Engine
The core of our revenue optimization strategy is a rule-based engine that adapts pricing in real-time based on market demand signals.

### Pricing Strategy Logic
The algorithm dynamically adjusts the `base_price` by calculating a premium or discount factor, ensuring we maximize revenue without sacrificing conversion rates:

- **Peak Demand (>20 calls/day):** A **1.8x premium** is applied. During these periods, our supply is limited, and we prioritize high-margin bookings.
- **High Demand (>15 calls/day):** A **1.4x uplift** is applied to capture surplus value while maintaining competitive conversion rates.
- **Low Demand (<8 calls/day):** A **0.7x discount** is triggered to stimulate interest and maintain operational utilization during slower periods.
- **Standard:** Maintains the baseline price for steady-state operations.



[Image of price elasticity of demand curve]


*Key Benefit: By automating pricing based on demand elasticity, we effectively smooth out the revenue curve, reducing volatility and ensuring optimal capacity utilization.*

In [4]:
def calculate_dynamic_price(demand, base_price):
    if demand > 20: return base_price * 1.8
    elif demand > 15: return base_price * 1.4
    elif demand < 8: return base_price * 0.7
    return base_price

## 5. Operational Decision Support
The `get_quote` function acts as the interface between our machine learning model and real-time business operations. It provides an immediate, data-driven recommendation during customer interactions.

### Workflow Integration
- **Input Aggregation**: The function accepts localized data (area, demand, and custom service baskets), mapping these inputs to the features learned during training.
- **Dynamic Inference**: It performs real-time inference using the trained XGBoost model to output the `closing_probability`—a critical metric for sales prioritization.
- **Pricing Recommendation**: By combining the model's predictive output with our `dynamic_price` business rules, it generates an optimized quote that balances revenue capture with conversion likelihood.

*Key Benefit: Empowering business owners to make consistent, data-backed pricing decisions on the fly, directly during the customer inquiry process.*

In [5]:
def get_quote(area, order_dict, demand):
    base_price = calculate_order_total(order_dict)

    input_data = pd.DataFrame({'demand': [demand], 'base_price': [base_price],
                               'month': [datetime.now().month], 'is_weekend': [datetime.now().weekday() >= 5],
                               'is_holiday': [0], 'area': [area], 'service_type': [list(order_dict.keys())[0]]})
    input_encoded = pd.get_dummies(input_data).reindex(columns=X.columns, fill_value=0)

    prob = model_xgbc.predict_proba(input_encoded)[0][1]
    dyn_price = calculate_dynamic_price(demand, base_price)

    print(f"--- הצעת מחיר ללקוח ---")
    print(f"מחיר בסיס: {base_price} ש\"ח | מחיר דינמי מומלץ: {dyn_price:.0f} ש\"ח")
    print(f"סיכויי סגירה משוערים: {prob*100:.1f}%")

my_order = {'sofa_l': 1, 'mattress_double': 2}
get_quote(area='Center', order_dict=my_order, demand=18)

--- הצעת מחיר ללקוח ---
מחיר בסיס: 1400 ש"ח | מחיר דינמי מומלץ: 1960 ש"ח
סיכויי סגירה משוערים: 74.6%


In [6]:
model_xgbc.save_model("pricing_model.json")